# Lecture 2: ReAct agents with LangGraph

## The big idea

Last lecture we built an agent from scratch. Its heart was a loop: call the LLM, check whether it asked for a tool, run the tool, send back the result, repeat.

That loop has a name: **ReAct**, short for **Rea**son + **Act** (Yao et al., 2022). The model alternates between three things until it can answer:

1. **Reason**: think about what to do next.
2. **Act**: request a tool call.
3. **Observe**: read the tool's result.

```
 question ──► Reason ──► Act ──► Observe ──┐
                ▲                          │
                └──────────────────────────┘
                │   (repeat until done)
                ▼
              answer
```

This lecture has two goals:

1. **Build a ReAct agent with LangGraph**, a library that describes an agent as a graph of steps. Same calculator as last week, but now using the API's built-in tool calling.
2. **Look at what is actually sent to the LLM on every turn.** This is the important part. The LLM has no memory. On every call, the harness sends the **entire conversation so far**. Each turn **appends** new messages and nothing is removed, so the context gets bigger, and more expensive, with every step.

## Step 0: Setup

You need an Anthropic API key in the `ANTHROPIC_API_KEY` environment variable.

We install LangGraph, LangChain's Anthropic integration and matplotlib for the charts.

In [ ]:
%pip install -q langgraph langchain langchain-anthropic matplotlib

In [ ]:
from langchain_anthropic import ChatAnthropic

MODEL = "claude-opus-5-5"

llm = ChatAnthropic(  # reads ANTHROPIC_API_KEY from the environment
    model=MODEL,
    max_tokens=16000,
    # If a safety classifier declines, retry on a fallback model automatically.
    betas=["server-side-fallback-2026-07-01"],
    model_kwargs={"fallbacks": "default"},
)

# Part 1: Building a ReAct agent with LangGraph

## Step 1: The tools are still just Python functions

Same four tools as last week. The only new thing is the `@tool` decorator. It reads the function's **name**, **type hints** and **docstring**, and builds a description of the tool that the model can understand. Last week we wrote that description by hand, in the system prompt.

In [ ]:
from langchain_core.tools import tool


@tool
def add(a: float, b: float) -> float:
    """Return a + b."""
    return a + b


@tool
def subtract(a: float, b: float) -> float:
    """Return a - b."""
    return a - b


@tool
def multiply(a: float, b: float) -> float:
    """Return a * b."""
    return a * b


@tool
def divide(a: float, b: float) -> float:
    """Return a / b."""
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b


TOOLS = [add, subtract, multiply, divide]

Here's the description `@tool` generated for `add`. This JSON schema is what gets sent to the API, in the `tools` field of the request.

In [ ]:
import json
from langchain_anthropic.chat_models import convert_to_anthropic_tool

print(json.dumps(convert_to_anthropic_tool(add), indent=2))

## Step 2: Built-in tool calling

`bind_tools` attaches the tool schemas to every request we send. The API now handles the "protocol" we invented last week: when the model wants a tool, the reply comes back with a structured list of `tool_calls`. No more JSON parsing on our side.

In [ ]:
from langchain_core.messages import HumanMessage

llm_with_tools = llm.bind_tools(TOOLS)

reply = llm_with_tools.invoke([HumanMessage("What is 48219 * 3172?")])

print("text:      ", repr(reply.text))
print("tool calls:", reply.tool_calls)

### What just happened?

Exactly what happened last week: **nothing was multiplied**. The model only *asked* for `multiply`. The request now arrives already parsed, but running the tool and sending back the result is still the harness's job.

Now look at how many tokens that one-line question used:

In [ ]:
print(reply.usage_metadata)

The question is only about a dozen tokens, but `input_tokens` is much bigger. The tool definitions, plus the instructions the API adds to explain tool use to the model, are part of the input too. They are sent **with every request**. Keep that in mind, we'll measure it in Part 2.

## Step 3: The state is a list of messages

A LangGraph agent passes a **state** from step to step. For a ReAct agent the state is just the conversation: a list of messages. LangGraph provides this ready-made as `MessagesState`.

Here's the important detail. A step never returns the whole list. It returns only its **new** messages, and a **reducer** function called `add_messages` appends them to the list. Let's call the reducer by hand to see what it does:

In [ ]:
from langchain_core.messages import AIMessage, ToolMessage
from langgraph.graph.message import add_messages

history = [HumanMessage("What is 2 + 3?")]
print(len(history), "message")

# The model's turn: it asks for a tool
tool_request = AIMessage("", tool_calls=[{"name": "add", "args": {"a": 2, "b": 3}, "id": "call_1"}])
history = add_messages(history, [tool_request])
print(len(history), "messages")

# The tool's turn: the result
history = add_messages(history, [ToolMessage("5.0", tool_call_id="call_1")])
print(len(history), "messages")

for message in history:
    message.pretty_print()

New messages go on the end of the list. Nothing is removed. (`add_messages` would only *replace* a message if you sent one with the same `id`, and our agent never does that.) Keep this in mind, it's the subject of Part 2.

## Step 4: The two nodes

A **node** is a plain Python function that receives the state and returns an update. A ReAct agent needs two:

- **`agent`** (Reason): sends the conversation to the LLM and returns its reply.
- **`tools`** (Act + Observe): runs the tools the model asked for and returns their results as `ToolMessage`s. LangGraph's prebuilt `ToolNode` does this. It's last week's `execute_tool`.

In [ ]:
from langchain_core.messages import SystemMessage
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode

SYSTEM_PROMPT = """You are an assistant that answers questions using tools. You are bad at
arithmetic, so you must never calculate anything yourself. Always use the arithmetic tools."""


def agent(state: MessagesState):
    """Reason: send the WHOLE conversation to the LLM and return its reply."""
    messages = [SystemMessage(SYSTEM_PROMPT)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    if response.response_metadata.get("stop_reason") == "refusal":
        raise RuntimeError("The model declined to answer")
    return {"messages": [response]}  # only the NEW message; add_messages appends it


# Act: run every tool call in the last AI message. Errors are sent back to the model as text.
tool_node = ToolNode(TOOLS, handle_tool_errors=True)

Look at the first line of `agent`:

```python
messages = [SystemMessage(SYSTEM_PROMPT)] + state["messages"]
```

Every time this node runs, it sends the system prompt **plus every message so far**. The LLM keeps no memory between calls, so this is the only way it can know what already happened.

## Step 5: Wire the nodes into a graph

**Edges** connect the nodes:

- Start at `agent`.
- After `agent`, a **conditional edge** looks at the reply. If it contains tool calls, go to `tools`. Otherwise, stop. The prebuilt `tools_condition` does this check. It's last week's `if call is None`.
- After `tools`, always go back to `agent`.

That cycle, `agent` → `tools` → `agent`, is the ReAct loop.

In [ ]:
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import tools_condition

builder = StateGraph(MessagesState)
builder.add_node("agent", agent)
builder.add_node("tools", tool_node)

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)  # tool calls? -> "tools", otherwise -> END
builder.add_edge("tools", "agent")

graph = builder.compile()

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:  # drawing the PNG needs internet access; fall back to the Mermaid source
    print(graph.get_graph().draw_mermaid())

### How this maps to last week's harness

| Lecture 1 (by hand) | Lecture 2 (LangGraph) |
|---|---|
| Tool descriptions written in `SYSTEM_PROMPT` | `@tool` + `bind_tools` |
| `parse_tool_call` reads JSON out of the text | Built-in tool calling: `response.tool_calls` |
| `execute_tool` | `ToolNode` |
| `if call is None: return reply` | `tools_condition` |
| `messages.append(...)` | The `add_messages` reducer |
| `for step in range(max_steps)` | The graph's cycle, with `recursion_limit` as the safety net |

## Step 6: Try it

`pretty_print()` would show the model's raw *thinking* blocks. This model returns them on every reply: they hold its private reasoning, hidden by default, and they stay inside the AI messages. This helper skips them and prints one line per message.

In [ ]:
ROLES = {"system": "SYSTEM", "human": "USER", "ai": "AI", "tool": "TOOL"}


def show_conversation(messages, width=110):
    """Print one line per message, skipping the model's hidden thinking blocks."""
    for message in messages:
        if isinstance(message, AIMessage) and message.tool_calls:
            text = ", ".join(
                call["name"] + "(" + ", ".join(f"{k}={v!r}" for k, v in call["args"].items()) + ")"
                for call in message.tool_calls
            )
        else:
            text = " ".join(message.text.split())  # squash newlines so it fits on one line
        if len(text) > width:
            text = text[:width] + f"... ({len(message.text):,} characters)"
        print(f"{ROLES[message.type]:<7} {text}")


result = graph.invoke({"messages": [HumanMessage("Add 12 and 30, multiply the result by 4, then divide by 7.")]})
show_conversation(result["messages"])

Each round of the ReAct loop is visible: an `AI` line with a tool request (Reason + Act), a `TOOL` line with the result (Observe), and a final `AI` line with the answer.

# Part 2: The context grows on every turn

The agent works. Now let's look at what it **sends**.

The LLM is stateless. Each call is independent and the model remembers nothing from the previous one. So on every call the harness has to send everything the model needs to know: the system prompt, the tool definitions and the whole conversation so far.

## Step 7: Watch the message list grow

`graph.stream(..., stream_mode="values")` gives us the full state after every step. Let's print the message list each time, using a short tag per message.

In [ ]:
def label(message):
    """A short tag for a message, so a whole conversation fits on one line."""
    if isinstance(message, SystemMessage):
        return "[system]"
    if isinstance(message, HumanMessage):
        return "[user]"
    if isinstance(message, ToolMessage):
        content = message.text
        return f"[tool: {content}]" if len(content) <= 20 else f"[tool: {len(content):,} chars]"
    if message.tool_calls:
        return "[ai → " + ", ".join(call["name"] for call in message.tool_calls) + "]"
    return "[ai: answer]"


question = "Add 12 and 30, multiply the result by 4, then divide by 7."
for state in graph.stream({"messages": [HumanMessage(question)]}, stream_mode="values"):
    messages = state["messages"]
    print(f"{len(messages)} messages: " + " ".join(label(m) for m in messages))

Each line is the previous line plus something new. And every time the `agent` node runs, it sends the **whole line**, plus the system prompt, to the LLM.

## Step 8: Measure what every LLM call sends

Let's add instrumentation. Below is the same agent as before, packaged as a function so we can reuse it with different tools. The only new lines are marked `# NEW`: after each LLM call, the `agent` node records which messages it sent and how many input tokens the API counted (`usage_metadata`).

In [ ]:
CALL_LOG = []  # one entry per LLM call


def build_agent(tools):
    """Steps 2, 4 and 5 in one function, plus a log of every LLM call."""
    llm_with_tools = llm.bind_tools(tools)

    def agent(state: MessagesState):
        messages = [SystemMessage(SYSTEM_PROMPT)] + state["messages"]
        response = llm_with_tools.invoke(messages)
        if response.response_metadata.get("stop_reason") == "refusal":
            raise RuntimeError("The model declined to answer")
        CALL_LOG.append({  # NEW: what this call sent, and what it cost
            "labels": [label(m) for m in messages],
            "input_tokens": response.usage_metadata["input_tokens"],
        })
        return {"messages": [response]}

    builder = StateGraph(MessagesState)
    builder.add_node("agent", agent)
    builder.add_node("tools", ToolNode(tools, handle_tool_errors=True))
    builder.add_edge(START, "agent")
    builder.add_conditional_edges("agent", tools_condition)
    builder.add_edge("tools", "agent")
    return builder.compile()


def run(graph, question):
    """Run the agent on a fresh conversation. Return the final state and the log of LLM calls."""
    CALL_LOG.clear()
    result = graph.invoke({"messages": [HumanMessage(question)]}, {"recursion_limit": 25})
    print("ANSWER:", result["messages"][-1].text, "\n")
    return result, list(CALL_LOG)


def show_calls(calls):
    """One line per LLM call: how much it sent, and how much of that was new."""
    previous = 0
    for i, call in enumerate(calls, start=1):
        tokens = call["input_tokens"]
        print(f"LLM call {i}: {len(call['labels'])} messages, {tokens:,} input tokens (+{tokens - previous:,} new)")
        print("    " + " ".join(call["labels"]))
        previous = tokens
    total = sum(call["input_tokens"] for call in calls)
    print(f"\nInput tokens billed over {len(calls)} calls: {total:,}")

In [ ]:
calculator = build_agent(TOOLS)

result, calc_calls = run(calculator, "Add 12 and 30, multiply the result by 4, then divide by 7.")
show_calls(calc_calls)

### What just happened?

- **Call 1** is the baseline: the system prompt, the tool definitions and the question.
- **Every later call re-sent everything from the previous call**, plus the new messages: the model's last tool request and the tool's result.
- The question was sent on **every** call. So were the system prompt and the tool definitions.
- We pay for the input tokens of every call, so the total billed is the **sum** of all the calls, much more than the size of the final conversation.

Here's the exact conversation sent on the **last** call. Call 1 sent only its first two lines.

In [ ]:
last_call = [SystemMessage(SYSTEM_PROMPT)] + result["messages"][:-1]  # everything except the final answer
show_conversation(last_call)

## Step 9: Picture it

Each bar is one LLM call. The blue part was already sent in the previous call. The orange part is new. Each bar contains the whole previous bar.

In [ ]:
import matplotlib.pyplot as plt

RESENT, NEW = "#2a78d6", "#eb6834"  # blue: sent again, orange: new in this call
INK, GRID = "#52514e", "#e1e0d9"


def plot_calls(calls, title):
    tokens = [call["input_tokens"] for call in calls]
    resent = [0] + tokens[:-1]  # what the previous call already sent
    new = [t - r for t, r in zip(tokens, resent)]
    x = range(1, len(tokens) + 1)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(x, resent, width=0.5, color=RESENT, edgecolor="white", linewidth=2, label="already sent in the previous call")
    ax.bar(x, new, width=0.5, bottom=resent, color=NEW, edgecolor="white", linewidth=2, label="new in this call")
    for xi, t in zip(x, tokens):
        ax.annotate(f"{t:,}", (xi, t), xytext=(0, 4), textcoords="offset points", ha="center", fontsize=9, color=INK)

    ax.set_title(title, loc="left", pad=28)
    ax.set_xlabel("LLM call")
    ax.set_ylabel("input tokens sent")
    ax.set_xticks(list(x))
    ax.yaxis.set_major_formatter(lambda value, _: f"{value:,.0f}")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color=GRID, linewidth=1)
    ax.set_axisbelow(True)
    ax.legend(frameon=False, loc="lower left", bbox_to_anchor=(0, 1), ncols=2)  # above the bars
    plt.show()


plot_calls(calc_calls, "Calculator agent: input tokens sent on each LLM call")

# Part 3: Big tool results make it worse

## Step 10: A tool that returns a lot of text

A calculator result is tiny: `42.0`. Real tools often return much more: a web page, a file, the rows of a database query. Let's add a tool that returns a whole year's sales report as a text table.

In [ ]:
import random

PRODUCTS = ["Laptops", "Monitors", "Keyboards", "Mice", "Headsets", "Webcams"]
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
REGIONS = ["North", "South", "East", "West"]


def make_report(region):
    """A made-up yearly report: units sold and revenue, per product and month."""
    rng = random.Random(region)  # same numbers every time for the same region
    header = "Month | " + " | ".join(PRODUCTS) + " | Total"
    units_rows, revenue_rows = [], []
    for month in MONTHS:
        units = [rng.randint(50, 900) for _ in PRODUCTS]
        revenue = [u * rng.randint(20, 120) for u in units]
        units_rows.append(f"{month} | " + " | ".join(f"{u:,}" for u in units) + f" | {sum(units):,}")
        revenue_rows.append(f"{month} | " + " | ".join(f"{r:,}" for r in revenue) + f" | {sum(revenue):,}")
    return "\n".join(
        [f"SALES REPORT: {region} region, 2026", "", "UNITS SOLD", header, *units_rows,
         "", "REVENUE (USD)", header, *revenue_rows]
    )


@tool
def get_sales_report(region: str) -> str:
    """Return the full 2026 sales report for a region: North, South, East or West."""
    if region.title() not in REGIONS:
        raise ValueError(f"Unknown region '{region}'. Known regions: {', '.join(REGIONS)}")
    return make_report(region.title())


report = get_sales_report.invoke({"region": "North"})
print(report)
print(f"\n{len(report):,} characters")

Now give the agent the calculator tools **and** the report tool, and ask a question that needs two reports.

In [ ]:
sales_agent = build_agent(TOOLS + [get_sales_report])

result, sales_calls = run(
    sales_agent,
    "What was the total revenue in Q3 (July to September) for the North region, "
    "and how much more or less was that than the South region?",
)
show_calls(sales_calls)

In [ ]:
plot_calls(sales_calls, "Sales agent: input tokens sent on each LLM call")

### What just happened?

Find the big orange block. That's the call right after the reports came back. From then on, the reports are part of **every** call, in blue, even though the model only needed six numbers from them (three monthly totals per region).

The report tool ran once per region. Its output was **sent** on every LLM call after that.

# Part 4: Why this matters

## Step 11: The context grows linearly, the bill grows quadratically

Say every call has a fixed part of **F** tokens (system prompt, tool definitions, question), and every step adds about **k** new tokens (a tool request and its result). Then:

- Call *n* sends **F + (n − 1)·k** tokens. The context grows **linearly**.
- After *n* calls we have been billed for **n·F + k·n(n − 1)/2** input tokens in total. The bill grows **quadratically**.

Doubling the number of steps roughly **quadruples** the input tokens billed. Let's take F and k from the calculator run and extrapolate to 50 steps, which is a modest task for a coding agent.

In [ ]:
F = calc_calls[0]["input_tokens"]                                    # the fixed part: call 1
k = (calc_calls[-1]["input_tokens"] - F) / (len(calc_calls) - 1)     # average growth per step
print(f"F = {F:,} tokens, k = {k:,.0f} tokens per step\n")

steps = list(range(1, 51))
context = [F + (n - 1) * k for n in steps]   # what call n sends
billed = [sum(context[:n]) for n in steps]   # total billed after n calls

for n in (5, 10, 25, 50):
    print(f"after {n:>2} steps: call {n} sends {context[n - 1]:>9,.0f} tokens, total billed {billed[n - 1]:>11,.0f}")

In [ ]:
CONTEXT_COLOR, BILLED_COLOR = "#2a78d6", "#eb6834"

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(steps, billed, color=BILLED_COLOR, linewidth=2, label="total input tokens billed so far")
ax.plot(steps, context, color=CONTEXT_COLOR, linewidth=2, label="tokens sent on this call (context size)")
for series, color in ((billed, BILLED_COLOR), (context, CONTEXT_COLOR)):
    ax.plot(steps[-1], series[-1], "o", color=color, markersize=7, markeredgecolor="white", markeredgewidth=2)
    ax.annotate(f"{series[-1]:,.0f}", (steps[-1], series[-1]), xytext=(-6, 8), textcoords="offset points",
                ha="right", fontsize=9, color=INK)

ax.set_title("Linear context, quadratic bill (extrapolated from the calculator run)", loc="left")
ax.set_xlabel("number of LLM calls")
ax.set_ylabel("tokens")
ax.yaxis.set_major_formatter(lambda value, _: f"{value:,.0f}")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color=GRID, linewidth=1)
ax.set_axisbelow(True)
ax.legend(frameon=False, loc="upper left")
plt.show()

### Why a growing context is a problem

- **Cost.** You pay for every input token of every call, so the total grows quadratically with the number of steps.
- **Speed.** The model has to read the whole context before it writes its first word. Longer context, slower replies.
- **Limits.** Every model has a maximum context window. A long-running agent will eventually hit it.
- **Quality.** The more irrelevant text there is in the context (like those full sales tables), the easier it is for the model to miss or mix up the details that matter.

### What can we do about it?

Every technique works on the same thing: **what goes into the message list**.

- **Smaller tool results.** Return only what the model needs (exercise 2).
- **Trimming.** Drop old messages before calling the LLM.
- **Summarizing.** Replace old turns with a short summary.
- **Clearing old tool results.** Keep the record that a tool was called, but drop its big output once it has been used.
- **Prompt caching.** Because the context is append-only, every request *starts with* the previous request. The provider can cache that shared beginning and charge much less to read it again. The context doesn't get smaller, but re-sending it gets cheaper and faster.

These trade off against each other. Trimming or summarizing changes the *start* of the context, so the next request no longer begins with the cached one.

Modern models add one more reason to keep the history append-only. Remember the hidden *thinking* blocks from Step 6? This model ties each one to the exact conversation that came before it. If you remove or edit earlier messages and then send the later thinking blocks back, the API can reject the request with an error saying the block is "bound to a different conversation". So client-side trimming also has to strip those thinking blocks. Alternatively, use the API's server-side compaction and tool-result clearing, which don't count as edits.

## Step 12: The one-liner

We built the graph by hand to see how it works. In practice, you'd use LangChain's prebuilt `create_agent`, which builds the same ReAct loop for you:

In [ ]:
from langchain.agents import create_agent

prebuilt = create_agent(llm, TOOLS, system_prompt=SYSTEM_PROMPT)

try:
    display(Image(prebuilt.get_graph().draw_mermaid_png()))
except Exception:
    print(prebuilt.get_graph().draw_mermaid())

Same shape: a `model` node, a `tools` node and the loop between them. `create_agent` also accepts **middleware**, hooks that run around every model call. That's where context-management techniques plug in, for example the prebuilt `SummarizationMiddleware`.

In [ ]:
show_conversation(prebuilt.invoke({"messages": [HumanMessage("What is 48219 * 3172?")]})["messages"])

## Key takeaways

- **ReAct is last week's loop with a name:** reason (LLM call), act (tool call), observe (tool result), repeat.
- **LangGraph makes the loop explicit:** a **state** (the list of messages), **nodes** (`agent` and `tools`) and **edges** (the conditional edge is the loop's exit).
- **The LLM is stateless.** The agent's "memory" is the message list, and the **whole list is sent on every call**.
- **The context grows on every turn.** Each step appends a tool request and its result. Nothing is removed.
- **The bill grows faster than the context.** Linear context means quadratic total input tokens. Big tool results make it much worse, because they are re-sent on every later call.

### Exercises

1. Ask the calculator agent "What is 10 divided by 0?". What does the `ToolMessage` contain? What happens if you build the `ToolNode` without `handle_tool_errors=True`?
2. Change `get_sales_report` so it returns only the monthly revenue totals. Re-run Step 10 and compare the chart and the total input tokens billed.
3. Change the `agent` node in `build_agent` so it sends only the system prompt, the question and the **last two** messages, then run the sales question. The API may reject the request because a thinking block is "bound to a different conversation" (see the end of Step 11). Remove the `thinking` blocks from the AI messages you send and try again. Does the agent still get the right answer? What did it forget?
4. Ask a question that needs no tools, like "What is the capital of France?", and look at `show_calls`. That's the fixed overhead of every call. Now build an agent with one tool fewer and measure again: how many tokens does one tool definition cost?